In [ ]:
import pymysql
from configparser import ConfigParser
# 讀取 .env 檔案取得資料庫連線資訊
config = ConfigParser()
config.read('../Chapter1/config.ini')

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

with connection.cursor() as cursor:
    sql = """
        CREATE DATABASE IF NOT EXISTS testdb;
    """
    # 執行建立的 SQL 語句
    cursor.execute(sql)
    # 執行查看資料庫
    cursor.execute("SHOW DATABASES;")
    result = cursor.fetchall()
    print(f"Databases: {result}")

    sql = """
        CREATE TABLE IF NOT EXISTS testdb.user (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            age INT NOT NULL,
            username VARCHAR(255) NOT NULL UNIQUE,
            password VARCHAR(255) NOT NULL
        );
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES FROM testdb;")
    result = cursor.fetchall()
    print(f"Tables: {result}")


In [ ]:
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database="testdb",
)

with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES ("Pochita", 10, "pochita", "123456")
    """
    cursor.execute(sql)

    cursor.execute("SELECT * FROM user")
    result = cursor.fetchall()
    for row in result:
        print(row)


In [ ]:
connection.commit() # 提交資料庫變更

In [ ]:
import random
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user (name, age, username, password)
        VALUES (%s, %s, %s, %s);
    """
    for i in range(10):
        # 隨機生成使用者名稱、年齡和密碼
        user = f'user_{i}'
        username = f'user_{i}'
        age = random.randint(18, 65)
        password = f'password_{random.randint(1000, 10000)}'
        cursor.execute(sql, (user, age, username, password))
    
    connection.commit()

##### ⚠️ <span style="color:red; font-weight:bold">勿使用 f-string 直接拼接 SQL 查詢</span>

In [ ]:
username = 'pochita'
password = '123456'

with connection.cursor() as cursor:
    # 查詢所有使用者
    sql = f"SELECT * FROM user WHERE username='{username}' and password='{password}';"
    cursor.execute(sql)
    result = cursor.fetchall()
    for row in result:
        print(f"\033[32m{row}\033[0m")

In [ ]:
# 使用 SQL Injection 攻擊，取得所有使用者的資料
username = "' OR 1=1; -- "
password = ""

with connection.cursor() as cursor:
    sql = f"""
        SELECT * FROM user WHERE username = '{username}' AND password = '{password}'
    """
    print(sql)
    cursor.execute(sql)
    result = cursor.fetchall()

    for row in result: 
        print(f"\033[31m{row}\033[0m")

In [ ]:
# 使用 SQL Injection 攻擊，取得所有使用者的資料
username = "' OR 1=1; -- "
password = ""

with connection.cursor() as cursor:
    sql = """
        SELECT * FROM user WHERE username = %s AND password = %s
    """
    print(sql)
    cursor.execute(sql, (username, password))
    result = cursor.fetchall()
    
print(f"\033[31m{result}\033[0m")

### 🚮清除資源

In [ ]:
with connection.cursor() as cursor:
    sql = """
        DROP DATABASE IF EXISTS testdb;
    """
    cursor.execute(sql)

    cursor.execute("SHOW DATABASES;")
    result = cursor.fetchall()
    print(f"Databases: {result}")